# 猫狗数据集

1、猫狗二分类数据集是一个压缩文件、需要进行解压操作
2、猫狗二分类数据集中图像尺寸大小不一致，需要调整到统一的尺寸
3、猫狗二分类数据集中仅8000张图像，数据较少，需要进行数据增强

# 解压数据

In [ ]:
import zipfile

# 指定zip文件路径
file_path = r'E:\03学习资料\coding_class\04 深度学习\25.02.15-猫狗二分类任务\dataset.zip'
# 指定解压路径，最好在当前目录下
extract_path = r'E:\03学习资料\coding_class\04 深度学习\25.02.15-猫狗二分类任务'

with zipfile.ZipFile(file_path,'r') as zip_:
    # 将文件解压到指定文件夹
    zip_.extractall(path = extract_path)


# 分类模型

In [6]:
import torch
import torch.nn  as nn
import torch.optim as optim
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt


class DogCat(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1 ,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1 ,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),     

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1 ,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)      

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(in_features=128*8*8, out_features=256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,1),
            nn.Sigmoid()

        )

    def forward(self,x):
        # 将图像传入卷积层
        x = self.conv_layers(x)
        # 将卷积层结果铺平
        x = x.view(x.size(0),-1)
        # 全连接层计算
        x = self.fc_layers(x)
        return x

# 数据增强

In [22]:
train_trans = transforms.Compose([
    # 统一数据尺寸大小
    transforms.Resize((64,64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomAffine(degrees=30, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

test_trans = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])

def convert_to_float(label):
    """
    将样本中的标签有long类型转化为float类型
    
    BCEloss要求标签类型为float类型，ImageFolder在处理标签时，为long类型
    """
    return torch.Tensor([float(label)])

# 用于处理图像的数据集类，根据子目录将图片分类
train_dataset = datasets.ImageFolder(r'E:\03学习资料\coding_class\04 深度学习\25.02.15-猫狗二分类任务\dataset\training_set',
                                    transform=train_trans,
                                    target_transform=convert_to_float)
test_dataset = datasets.ImageFolder(r'E:\03学习资料\coding_class\04 深度学习\25.02.15-猫狗二分类任务\dataset\test_set',
                                    transform=test_trans,
                                    target_transform=convert_to_float)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [34]:
next(iter(train_loader))[1].size()

torch.Size([32, 1])

# 模型训练

In [36]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 100
model = DogCat().to(device)
criterion = nn.BCELoss() # 定义损失函数
optimizer = optim.Adam(model.parameters()) # 梯度函数

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_correct = 0 # 正确的数量
    for data, target in train_loader:
        data,target = data.to(device),target.to(device)
        # 梯度归领
        optimizer.zero_grad()
        # 前向传播，并把结果铺平
        output = model(data)
        # 计算损失
        loss = criterion(output,target)
        # 反向传播
        loss.backward()
        optimizer.step()
        # 计算当前批次损失
        train_loss += loss.item()
        # 计算预算结果
        y_pred = (output > 0.5).float()
        # 计算预测正确结果数量
        train_correct += (y_pred == target).sum().item()

    train_loss /= len(train_loader)
    train_accuracy = train_correct/len(train_loader.dataset)


    model.eval()
    test_loss = 0
    test_correct = 0
    with torch.no_grad():
        for data,target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output,target)
            test_loss += loss.item()
            y_pred = (output > 0.5).float()
            test_correct = (y_pred == target).sum().item()

        test_loss /= len(test_loader)
        test_accuracy = test_correct/len(test_loader.dataset)
    print(f'eopch:{epoch+1}/{epochs}, train_loss:{train_loss:.4f}, train_accuracy:{train_accuracy:.4f}, test_loss:{test_loss:.4f}, test_accuracy:{test_accuracy:.4f}')


eopch:1/100, train_loss:0.7534, train_accuracy:0.5249, test_loss:0.6716, test_accuracy:0.0065
eopch:2/100, train_loss:0.6777, train_accuracy:0.5764, test_loss:0.6618, test_accuracy:0.0050
eopch:3/100, train_loss:0.6605, train_accuracy:0.6045, test_loss:0.6485, test_accuracy:0.0040
eopch:4/100, train_loss:0.6569, train_accuracy:0.6076, test_loss:0.6102, test_accuracy:0.0055
eopch:5/100, train_loss:0.6477, train_accuracy:0.6184, test_loss:0.6643, test_accuracy:0.0045
eopch:6/100, train_loss:0.6397, train_accuracy:0.6346, test_loss:0.5957, test_accuracy:0.0060
eopch:7/100, train_loss:0.6335, train_accuracy:0.6430, test_loss:0.5938, test_accuracy:0.0055
eopch:8/100, train_loss:0.6281, train_accuracy:0.6490, test_loss:0.5692, test_accuracy:0.0070
eopch:9/100, train_loss:0.6214, train_accuracy:0.6539, test_loss:0.5591, test_accuracy:0.0070
eopch:10/100, train_loss:0.6161, train_accuracy:0.6605, test_loss:0.6303, test_accuracy:0.0045
eopch:11/100, train_loss:0.6085, train_accuracy:0.6701, tes

In [40]:
# 通常会用 .pt 或者 .pth 后缀来保存模型。
model_path = r'E:\LMAI_study\06 深度学习练习\model\cat_dog_classfiy.pth'
torch.save(model.state_dict(),model_path)

In [42]:
# 加载预训练模型
model = DogCat()
model.load_state_dict(torch.load(model_path))
model.eval()

DogCat(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=8192, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_